# Calories log1p ablation

Quick compare: does `log1p(calories_sum)` improve **linear_regression** or **ordered_logistic** MAE?

- Same participant hold-out and GroupKFold CV
- Only `calories_sum` differs between variants (all other features from processed CSV)
- Separate Optuna tune per (model, variant), `n_trials=10`

In [2]:
%pip install -q -r ../../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", message="IProgress not found")

_src = Path('../../src').resolve()
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

for _mod in [k for k in list(sys.modules) if k == 'modeling' or k.startswith('modeling.')]:
    del sys.modules[_mod]

from modeling.config import DATA_PATH, N_CV_FOLDS, RANDOM_STATE
from modeling.cv import run_model_benchmark
from modeling.data import load_fatigue_data, prepare_splits
from modeling.registry import ORDINAL_MODELS, get_search_space, make_model_factory
from modeling.tuning import tune_model

N_TRIALS = 10
MODELS = ['linear_regression', 'ordered_logistic']
VARIANTS = [
    ('baseline', False),
    ('log1p_calories', True),
]

In [4]:
df = load_fatigue_data('../../' + DATA_PATH)
bundle = prepare_splits(df)

print(f"Rows: {len(df):,}  Participants: {df['id'].nunique()}")
print(f"Train/val rows: {len(bundle.y_ord_train_val):,}  Test rows: {len(bundle.y_ord_test):,}")
print(f"Test participant ids: {sorted(bundle.test_ids)}")

Rows: 3,331  Participants: 42
Train/val rows: 2,646  Test rows: 685
Test participant ids: [np.int64(10), np.int64(18), np.int64(30), np.int64(37), np.int64(38), np.int64(42), np.int64(46), np.int64(50)]


In [5]:
def calories_feature_matrices(bundle, log1p_calories=False):
    """Pipeline feature matrices; optionally log1p only calories_sum."""
    X_train_val = bundle.X_train_val.copy()
    X_test = bundle.X_test.copy()
    if log1p_calories:
        X_train_val['calories_sum'] = np.log1p(X_train_val['calories_sum'])
        X_test['calories_sum'] = np.log1p(X_test['calories_sum'])
    return X_train_val, X_test

In [6]:
results = []

for variant_name, use_log1p in VARIANTS:
    X_train_val, X_test = calories_feature_matrices(bundle, log1p_calories=use_log1p)
    for name in MODELS:
        print(f"Tuning {name} ({variant_name})...")
        params, cv_score = tune_model(
            name=name,
            registry=ORDINAL_MODELS,
            search_space=get_search_space(name),
            y_train_val=bundle.y_ord_train_val,
            groups=bundle.groups_train_val,
            X_train_val=X_train_val,
            n_trials=N_TRIALS,
            n_splits=N_CV_FOLDS,
            test_ids=bundle.test_ids,
        )
        factory = make_model_factory(name, params, ORDINAL_MODELS)
        bench = run_model_benchmark(
            name=name,
            model_factory=factory,
            n_splits=N_CV_FOLDS,
            test_ids=bundle.test_ids,
            X_train_val=X_train_val,
            X_test=X_test,
            y_train_val=bundle.y_ord_train_val,
            y_test=bundle.y_ord_test,
            groups=bundle.groups_train_val,
        )
        results.append({
            'variant': variant_name,
            'model': name,
            'best_params': params,
            'cv_mae': bench['cv_summary'].loc['mean', 'mae'],
            'test_mae': bench['test_metrics']['mae'],
            'test_qwk': bench['test_metrics']['qwk'],
        })
        print(
            f"  [ok] cv_mae={results[-1]['cv_mae']:.4f}  "
            f"test_mae={results[-1]['test_mae']:.4f}"
        )

results_df = pd.DataFrame(results)
results_df

Tuning linear_regression (baseline)...
  [ok] cv_mae=1.4001  test_mae=1.2599
Tuning ordered_logistic (baseline)...
  [ok] cv_mae=1.4725  test_mae=1.3036
Tuning linear_regression (log1p_calories)...
  [ok] cv_mae=1.4016  test_mae=1.2584
Tuning ordered_logistic (log1p_calories)...
  [ok] cv_mae=1.4814  test_mae=1.3007


,variant,model,best_params,cv_mae,test_mae,test_qwk
0,baseline,linear_regression,{'alpha': 0.9807504051673361},1.400118,1.259854,-0.112465
1,baseline,ordered_logistic,{'alpha': 4.343550939104513},1.472520,1.303650,-0.128929
2,log1p_calories,linear_regression,{'alpha': 3.226915769290616},1.401597,1.258394,-0.110094
3,log1p_calories,ordered_logistic,{'alpha': 1.38704887501834},1.481439,1.300730,-0.125785


In [7]:
summary = results_df.pivot(index='model', columns='variant', values=['cv_mae', 'test_mae', 'test_qwk'])
summary.columns = [f"{metric}_{variant}" for metric, variant in summary.columns]
summary['delta_test_mae'] = summary['test_mae_log1p_calories'] - summary['test_mae_baseline']
summary['delta_cv_mae'] = summary['cv_mae_log1p_calories'] - summary['cv_mae_baseline']

print('Summary (delta < 0 means log1p(calories_sum) helped)')
display(summary.sort_index())

for model in MODELS:
    delta = summary.loc[model, 'delta_test_mae']
    direction = 'helped' if delta < 0 else 'did not help' if delta > 0 else 'no change'
    print(f"{model}: log1p calories {direction} (delta test MAE {delta:+.4f})")

Summary (delta < 0 means log1p(calories_sum) helped)


,cv_mae_baseline,cv_mae_log1p_calories,test_mae_baseline,test_mae_log1p_calories,test_qwk_baseline,test_qwk_log1p_calories,delta_test_mae,delta_cv_mae
model,,,,,,,,
linear_regression,1.400118,1.401597,1.259854,1.258394,-0.112465,-0.110094,-0.00146,0.001479
ordered_logistic,1.472520,1.481439,1.303650,1.300730,-0.128929,-0.125785,-0.00292,0.008918


linear_regression: log1p calories helped (delta test MAE -0.0015)
ordered_logistic: log1p calories helped (delta test MAE -0.0029)


## Interpretation

NO significant improve is found. No need to log-transform calories_sum.